# Immune Data Embeddings

In [1]:
from config import ScImmuneConfig
from model import ScImmuneModel
from tokenizer import ScImmuneTokenizer # refactored version

import torch
import os
import shutil
from utils import generate_metadata_embeddings, generate_metadata_tokens, assign_ontology_embeddings
from gensim.models import Word2Vec
import anndata as ad
import pandas as pd
import scanpy as sc
import json

/home/s5srinivasan/immune-foundational-model/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Modify tokens and embeddings

In [8]:
shutil.copy("config.json", "scimmune-model/config.json")
shutil.copy("og_model.bin", "scimmune-model/pytorch_model.bin")

'scimmune-model/pytorch_model.bin'

In [6]:
local_config = ScImmuneConfig.from_pretrained("scimmune-model") # load config locally
local_model = ScImmuneModel(local_config) # load model locally
local_tokenizer = ScImmuneTokenizer(vocab_file="vocab_with_metadata.json") # initialize tokenizer

In [7]:
len(local_tokenizer) # 350 new metadata tokens -> 60698 + 350 = 61048
new_vocab_len = len(local_tokenizer)

In [5]:
new_vocab_len

61048

In [9]:
local_model.resize_token_embeddings(new_vocab_len)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(61048, 512, padding_idx=0)

In [10]:
local_model # inspect full model

ScImmuneModel(
  (gene_encoder): GeneEncoder(
    (embedding): Embedding(61048, 512, padding_idx=0)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (lin1): Linear(in_features=1, out_features=512, bias=True)
    (act): ReLU()
    (lin2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (no

## Re-initialize metadata tokens with Node2Vec vectors

In [7]:
embedding_layer = local_model.get_input_embeddings()
embedding_layer

Embedding(61048, 512, padding_idx=0)

In [8]:
# Set Node2Vec model folder path

n2vmodel_path = "../utils/obo_models"

# Load all embedding vectors
doid_embeddings = Word2Vec.load(f"{n2vmodel_path}/doid_node2vec.model") # disease
cl_embeddings = Word2Vec.load(f"{n2vmodel_path}/cl_node2vec.model") # cell type
hancestro_embeddings = Word2Vec.load(f"{n2vmodel_path}/hancestro_node2vec.model") # ethinicity
hsapdv_embeddings = Word2Vec.load(f"{n2vmodel_path}/hsapdv_node2vec.model") # human development
pato_embeddings = Word2Vec.load(f"{n2vmodel_path}/pato_node2vec.model") # sex
uberon_embeddings = Word2Vec.load(f"{n2vmodel_path}/uberon_node2vec.model") # tissue

In [9]:
with open("vocab_with_metadata.json", "r") as f:
    vocab_with_metadata_dict = json.load(f) # load this as a dict for lookup

In [10]:
tag2ontology_map = {
    "disease" : "doid",
    "cell_type" : "cl",
    "self_reported_ethnicity" : "hancestro",
    "development_stage" : "hsapdv",
    "sex" : "pato",
    "tissue_general" : "uberon",
}

In [11]:
# Assign cell type ontology embeddings
for tag, ontology in tag2ontology_map.items():
    assign_ontology_embeddings(
        tokenizer=local_tokenizer,
        model=local_model,
        node2vec_model_path=f"{n2vmodel_path}/{ontology}_node2vec.model",
        tag=f"{tag}"
        )

[INFO] Assigned 1 Node2Vec embeddings for tag <disease=...>
[WARN] No Node2Vec embedding found for CL:0010003, skipping.
[WARN] No Node2Vec embedding found for unknown, skipping.
[INFO] Assigned 226 Node2Vec embeddings for tag <cell_type=...>
[WARN] No Node2Vec embedding found for unknown, skipping.
[INFO] Assigned 14 Node2Vec embeddings for tag <self_reported_ethnicity=...>
[INFO] Assigned 99 Node2Vec embeddings for tag <development_stage=...>
[INFO] Assigned 2 Node2Vec embeddings for tag <sex=...>
[INFO] Assigned 5 Node2Vec embeddings for tag <tissue_general=...>


In [12]:
# Check at random 
# doid_embeddings.wv["DOID:4"]
token = "<disease=DOID:4>"
oid = token[1:-1].split("=")[1] # get OID
token_id = local_tokenizer.convert_tokens_to_ids(token)

print(token)
print(token_id)

n2v_embedding = doid_embeddings.wv[oid]

model_embedding = embedding_layer.weight.data[token_id]

<disease=DOID:4>
61045


In [ ]:
model_embedding

In [ ]:
n2v_embedding

In [ ]:
local_model.save_pretrained("scImmune_metadata_model")

: 